# Phase 2: Tokenizer Training & Setup (SentencePiece)

This notebook covers:
1. **Locating & Loading Data:** Robust path handling to find `train.tsv` and `valid.tsv` (supports running from repo root, `models/`, or Google Colab).
2. **Preparing the Training Corpus:** Extracting all source sentences and target questions into `corpus.txt`.
3. **Training SentencePiece Tokenizer:** Training a Unigram subword model with vocab size 8000, 100% character coverage, and atomic `<ans>`, `</ans>` special tokens.
4. **Verification & Testing:** Checking that `<ans>` tags are preserved as single tokens, testing encoding/decoding, and subword fertility.

In [ ]:
# Install dependencies if running in a fresh environment
%pip install -q sentencepiece transformers datasets

In [ ]:
import os
import csv
import sentencepiece as spm

ANS_OPEN = "<ans>"
ANS_CLOSE = "</ans>"

# Ensure output models directory exists
MODEL_DIR = "models" if os.path.basename(os.getcwd()) != "models" else "."
os.makedirs(MODEL_DIR, exist_ok=True)
MODEL_PREFIX = os.path.join(MODEL_DIR, "ur_sp")

### 1. Robust Data Path Resolution
Automatically finds `train.tsv` and `valid.tsv` whether running from:
- Workspace root (`data/train.tsv`)
- Inside `models/` directory (`../data/train.tsv`)
- Google Colab (`/content/data/train.tsv` or `/content/train.tsv`)

In [ ]:
def find_data_file(filename):
    candidates = [
        os.path.join("data", filename),             # Running from repo root
        os.path.join("..", "data", filename),         # Running from inside models/
        os.path.join("/content", "data", filename), # Google Colab data folder
        os.path.join("/content", filename),        # Google Colab root
        filename,                                  # Current working directory
    ]
    for path in candidates:
        if os.path.exists(path):
            print(f"Found {filename} at: {os.path.abspath(path)}")
            return path
    raise FileNotFoundError(f"Could not find {filename} in any candidate location: {candidates}")

train_path = find_data_file("train.tsv")
valid_path = find_data_file("valid.tsv")

### 2. Load TSV Pairs

In [ ]:
def load_tsv_pairs(path):
    pairs = []
    with open(path, "r", encoding="utf-8") as f:
        reader = csv.reader(f, delimiter="\t", quoting=csv.QUOTE_NONE, escapechar="\\")
        for row in reader:
            if len(row) == 2:
                pairs.append((row[0], row[1]))
    print(f"Loaded {len(pairs):,} pairs from {path}")
    return pairs

train_pairs = load_tsv_pairs(train_path)
valid_pairs = load_tsv_pairs(valid_path)

### 3. Generate Plain-Text Corpus for SentencePiece
SentencePiece requires raw sentences (one per line) in a text file.

In [ ]:
corpus_path = os.path.join(MODEL_DIR, "corpus.txt") if MODEL_DIR != "." else "corpus.txt"

with open(corpus_path, "w", encoding="utf-8") as f:
    for src, tgt in train_pairs:
        f.write(src + "\n" + tgt + "\n")

print(f"Wrote {len(train_pairs) * 2:,} sentences to {corpus_path}")

### 4. Train SentencePiece Tokenizer

In [ ]:
spm.SentencePieceTrainer.train(
    input=corpus_path,
    model_prefix=MODEL_PREFIX,
    vocab_size=8000,
    model_type="unigram",
    character_coverage=1.0,                    # 100% coverage for full Urdu alphabet
    user_defined_symbols=[ANS_OPEN, ANS_CLOSE], # Prevent splitting <ans> and </ans>
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3,
)

print(f"SentencePiece model trained successfully! Saved to {MODEL_PREFIX}.model and {MODEL_PREFIX}.vocab")

### 5. Verify & Test Trained Tokenizer

In [ ]:
sp = spm.SentencePieceProcessor()
sp.load(f"{MODEL_PREFIX}.model")

print(f"Loaded tokenizer. Vocab size: {sp.get_piece_size()}")
print(f"{ANS_OPEN} ID: {sp.piece_to_id(ANS_OPEN)}")
print(f"{ANS_CLOSE} ID: {sp.piece_to_id(ANS_CLOSE)}")

# Test tokenization on a sample from the training set
sample_src, sample_tgt = train_pairs[0]
print("\n--- Sample Source ---")
print(sample_src)

pieces = sp.encode_as_pieces(sample_src)
ids = sp.encode_as_ids(sample_src)

print("\n--- Subword Pieces (first 25) ---")
print(pieces[:25])

# Verify <ans> and </ans> are kept as atomic single tokens
assert ANS_OPEN in pieces, f"{ANS_OPEN} was split into subwords!"
assert ANS_CLOSE in pieces, f"{ANS_CLOSE} was split into subwords!"
print("\nSUCCESS: <ans> and </ans> are preserved as single tokens!")

# Verify decoding integrity
decoded = sp.decode(ids)
print("\n--- Decoded Text ---")
print(decoded)
assert decoded == sample_src, "Decoded text does not match original!"
print("\nSUCCESS: Perfect lossless encoding and decoding!")